# Module 4 — Variational Autoencoder: looking inside

**Goal.** Turn the autoencoder (AE) from Module 2 into a variational autoencoder (VAE), then inspect the computations that make it generative.

We keep **MNIST**, the **2D latent space**, and the same hidden-layer sizes:

\[
\text{AE:}\qquad 784\to128\to2\to128\to784
\]

\[
\text{VAE:}\qquad 784\to128\to (\mu\in\mathbb R^2,\ \log\sigma^2\in\mathbb R^2)\to z\in\mathbb R^2\to128\to784
\]

The VAE has **two encoder output heads** instead of one. The decoder remains `2 → 128 → 784`.

**What you will inspect:** the 128 hidden activations, encoder means and standard deviations, random latent samples, gradients through sampling, reconstruction/KL losses, 2D distributions, generated digits, and interpolations.

Run the cells in order. The notebook runs in Colab or in a local Python environment with PyTorch, torchvision, NumPy, and Matplotlib. MNIST is downloaded on first use. Class labels are used **only for visualization**, never for training.

## 1. Same data and training setup as the AE

We reuse the same random seed, training subset, test subset, batch size, learning rate, and epoch count. That makes the AE–VAE comparison more informative. A CPU is sufficient; a GPU may make training faster.

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_N = 20_000
TEST_N = 3_000
BATCH_SIZE = 256
EPOCHS = 10
LR = 0.001
BETA = 1.0  # 1.0 = standard VAE objective; try other values later
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

transform = transforms.ToTensor()  # pixel values in [0, 1]
train_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_full = datasets.MNIST('./data', train=False, download=True, transform=transform)

rng = torch.Generator().manual_seed(SEED)
train_ids = torch.randperm(len(train_full), generator=rng)[:TRAIN_N]
test_ids = torch.randperm(len(test_full), generator=rng)[:TEST_N]
train_ds = Subset(train_full, train_ids.tolist())
test_ds = Subset(test_full, test_ids.tolist())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

fixed_x, fixed_y = next(iter(test_loader))
fixed_x, fixed_y = fixed_x[:8].to(DEVICE), fixed_y[:8]
print(f'Train: {len(train_ds):,} images | Test: {len(test_ds):,} images')

## 2. Replace the AE encoder with a probabilistic encoder

In our AE, `enc2(hidden)` returned one point `(z1, z2)`.

In the VAE, **the same 128 hidden activations feed two linear layers**:

- `enc_mu(hidden)` returns two **means** \(\mu=(\mu_1,\mu_2)\).
- `enc_logvar(hidden)` returns two **log-variances** \(\log\sigma^2\). We store these rather than \(\sigma\) because a log-variance may be any real number and produces a positive variance after exponentiation.

We then sample a point \(z\) from that Gaussian distribution. We keep both decoder layers unchanged from the AE.

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        # AE had enc1 and enc2. VAE replaces enc2 with two heads.
        self.enc1 = nn.Linear(784, 128)
        self.enc_mu = nn.Linear(128, 2)
        self.enc_logvar = nn.Linear(128, 2)

        # Identical decoder architecture to the AE.
        self.dec1 = nn.Linear(2, 128)
        self.dec2 = nn.Linear(128, 784)

    def encode(self, x):
        x = x.reshape(x.size(0), 784)
        h = torch.relu(self.enc1(x))
        mu = self.enc_mu(h)
        logvar = self.enc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)  # fresh N(0,I) noise
        return mu + std * eps

    def decode(self, z):
        h = torch.relu(self.dec1(z))
        x_hat = torch.sigmoid(self.dec2(h))
        return x_hat.reshape(-1, 1, 28, 28)

    def forward(self, x, sample=True):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar) if sample else mu
        return self.decode(z), mu, logvar, z

model = VAE().to(DEVICE)
print(model)

**Look inside the tensor shapes.** An input batch still contains 784 pixels per image. Only the second encoder layer has changed. We now output **two means and two log-variances** for every image, not a four-dimensional latent code.

In [ ]:
with torch.no_grad():
    batch = fixed_x
    flat = batch.reshape(len(batch), 784)
    hidden = torch.relu(model.enc1(flat))
    mu, logvar = model.encode(batch)
    std = torch.exp(0.5 * logvar)
    z = model.reparameterize(mu, logvar)
    output = model.decode(z)

for name, tensor in [('Pixels', flat), ('Encoder hidden', hidden),
                     ('Mean mu', mu), ('Log-variance', logvar),
                     ('Std dev', std), ('Sample z', z),
                     ('Reconstruction', output)]:
    print(f'{name:17s}: {tuple(tensor.shape)}')
print('\nFirst digit (before training):')
print('mu =', np.round(mu[0].cpu().numpy(), 3))
print('std =', np.round(std[0].cpu().numpy(), 3))
print('z =', np.round(z[0].cpu().numpy(), 3))
assert mu.shape == std.shape == z.shape == (8, 2)

## 3. Look inside sampling: the reparameterization trick

The encoder does not directly choose \(z\). It chooses the parameters of a distribution. The VAE samples using

\[
\epsilon\sim\mathcal N(0,I),\qquad z=\mu+\sigma\odot\epsilon.
\]

Randomness is contained in \(\epsilon\); the rest is a differentiable calculation. This allows gradients to reach both encoder heads. Inspect the numbers and gradient values below.

In [ ]:
mu_demo = torch.tensor([[1.0, -0.5]], requires_grad=True)
logvar_demo = torch.tensor([[np.log(0.2**2), np.log(0.7**2)]],
                           dtype=torch.float32, requires_grad=True)
eps_demo = torch.tensor([[0.5, -1.0]])  # fixed noise for a reproducible example
std_demo = torch.exp(0.5 * logvar_demo)
z_demo = mu_demo + std_demo * eps_demo

print('mu:      ', mu_demo.detach().numpy().round(3))
print('std:     ', std_demo.detach().numpy().round(3))
print('epsilon: ', eps_demo.numpy())
print('z:       ', z_demo.detach().numpy().round(3))

# A small demonstration objective: gradients flow through the sample.
example_objective = (z_demo ** 2).sum()
d_mu, d_logvar = torch.autograd.grad(example_objective, (mu_demo, logvar_demo))
print('Gradient w.r.t. mu:     ', d_mu.detach().numpy().round(3))
print('Gradient w.r.t. logvar: ', d_logvar.detach().numpy().round(3))

## 4. What do we optimize? Two losses

For each input image, the VAE tries to reconstruct it from a **random sample** of its encoded distribution. It also penalizes deviation from the chosen prior \(p(z)=\mathcal N(0,I)\).

\[
\underbrace{L_{\rm recon}}_{\text{recover input}}\; +\;
\beta\underbrace{D_{\rm KL}(q_\phi(z\mid x)\,\|\,\mathcal N(0,I))}_{\text{stay near the prior}}.
\]

We use a **binary cross-entropy reconstruction score**, summed across the 784 pixels, to match the common MNIST VAE example from the lecture. For genuinely binary pixels it is a Bernoulli negative log-likelihood. Because our MNIST pixels include intermediate grayscale intensities, here it is a practical *soft-target cross-entropy* rather than an exact likelihood for continuous pixel values.

For a diagonal Gaussian encoder, the KL term has an exact formula. We compute the **two terms separately** so we can see what training changes. `BETA=1` gives the standard relative weighting; changing it is an experiment, not a new definition of the original VAE.

In [ ]:
def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    # One reconstruction score per image; 784 pixel scores are added.
    recon_each = F.binary_cross_entropy(x_hat, x, reduction='none')
    recon_each = recon_each.flatten(1).sum(dim=1)

    # KL(q(z|x) || N(0,I)); sum over the two latent dimensions.
    kl_each = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1)

    total = (recon_each + beta * kl_each).mean()
    return total, recon_each.mean(), kl_each.mean()

with torch.no_grad():
    x_hat, mu, logvar, z = model(fixed_x)
    loss0, rec0, kl0 = vae_loss(fixed_x, x_hat, mu, logvar, beta=BETA)
print(f'Before training: reconstruction={rec0.item():.2f}, KL={kl0.item():.2f}, total={loss0.item():.2f}')

**Why does KL encourage a common distribution?** It is zero when the encoder's Gaussian equals `N(0,I)`. Moving its mean away from zero or making its spread much narrower/wider increases the penalty. **But** reconstruction needs the representations of different inputs to remain informative. The two objectives compete.

In [ ]:
# Three examples of diagonal 2D Gaussian distributions.
toy_mu = torch.tensor([[0., 0.], [2., 0.], [0., 0.]])
toy_std = torch.tensor([[1., 1.], [1., 1.], [0.15, 0.15]])
toy_logvar = 2 * toy_std.log()
toy_kl = -0.5 * (1 + toy_logvar - toy_mu.square() - toy_logvar.exp()).sum(1)
for label, k in zip(['mu=(0,0), std=(1,1)', 'mu=(2,0), std=(1,1)',
                     'mu=(0,0), std=(.15,.15)'], toy_kl):
    print(f'{label:27s}  KL={k.item():.3f}')

## 5. Train the VAE

The training loop is the same as in the AE notebook: forward pass, loss, `backward()`, optimizer step. The changes are **sampling in the forward pass** and **adding the KL term**. We also record both losses separately. For inspection, reconstruction snapshots decode each image from its **mean** `mu` (no randomness), so images from different epochs are comparable.

In [ ]:
def show_reconstructions(source, recon, title=''):
    source = source.detach().cpu().numpy()
    recon = recon.detach().cpu().numpy()
    n = len(source)
    fig, ax = plt.subplots(2, n, figsize=(1.4*n, 3.2))
    for i in range(n):
        ax[0, i].imshow(source[i, 0], cmap='gray_r', vmin=0, vmax=1)
        ax[1, i].imshow(recon[i, 0], cmap='gray_r', vmin=0, vmax=1)
        ax[0, i].axis('off')
        ax[1, i].axis('off')
    ax[0, 0].set_ylabel('Input', fontsize=11)
    ax[1, 0].set_ylabel('Output', fontsize=11)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

with torch.no_grad():
    before_training = model(fixed_x, sample=False)[0].cpu()
show_reconstructions(fixed_x, before_training, 'Before training (decode encoder means)')

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = {'train_total': [], 'train_recon': [], 'train_kl': [],
           'test_total': [], 'test_recon': [], 'test_kl': [], 'test_mse': []}
snapshots = {0: before_training.numpy()}

@torch.no_grad()
def evaluate(loader):
    model.eval()
    totals = np.zeros(4, dtype=float)  # loss, reconstruction, KL, pixel MSE
    count = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        x_hat, mu, logvar, z = model(x, sample=True)
        loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=BETA)
        # Report deterministic mean-based pixel MSE separately, as in the AE.
        deterministic = model.decode(mu)
        mse = F.mse_loss(deterministic, x).item()
        totals += len(x) * np.array([loss.item(), recon.item(), kl.item(), mse])
        count += len(x)
    return totals / count

for epoch in range(1, EPOCHS + 1):
    model.train()
    totals = np.zeros(3, dtype=float)
    count = 0
    for x, _ in train_loader:
        x = x.to(DEVICE)
        x_hat, mu, logvar, z = model(x, sample=True)
        loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=BETA)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        totals += len(x) * np.array([loss.item(), recon.item(), kl.item()])
        count += len(x)

    train_values = totals / count
    test_values = evaluate(test_loader)
    for key, value in zip(['train_total', 'train_recon', 'train_kl'], train_values):
        history[key].append(value)
    for key, value in zip(['test_total', 'test_recon', 'test_kl', 'test_mse'], test_values):
        history[key].append(value)

    if epoch in {1, 3, EPOCHS // 2, EPOCHS}:
        model.eval()
        with torch.no_grad():
            snapshots[epoch] = model(fixed_x, sample=False)[0].cpu().numpy()

    print(f'Epoch {epoch:2d}/{EPOCHS} | train recon={train_values[1]:6.2f} '
          f'KL={train_values[2]:5.2f} | test MSE={test_values[3]:.4f}')

## 6. Look inside training: loss terms and reconstructions

The total loss is `reconstruction + BETA × KL`; the two terms are not supposed to be equal. A lower KL does **not** automatically mean a better VAE: the encoder could discard information entirely. Compare the curves and the fixed digits reconstructed at several epochs.

In [ ]:
epoch_axis = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
axes[0].plot(epoch_axis, history['train_recon'], 'o-', label='Train')
axes[0].plot(epoch_axis, history['test_recon'], 'o-', label='Test')
axes[0].set(title='Reconstruction score', xlabel='Epoch', ylabel='Sum per image')
axes[1].plot(epoch_axis, history['train_kl'], 'o-', label='Train')
axes[1].plot(epoch_axis, history['test_kl'], 'o-', label='Test')
axes[1].set(title='KL divergence', xlabel='Epoch', ylabel='Sum per image')
axes[2].plot(epoch_axis, history['test_mse'], 'o-')
axes[2].set(title='Test reconstruction MSE', xlabel='Epoch', ylabel='Mean per pixel')
for ax in axes:
    ax.grid(alpha=.2)
axes[0].legend(frameon=False)
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
stages = sorted(snapshots)
fig, ax = plt.subplots(len(stages)+1, len(fixed_x),
                       figsize=(11, 1.45*(len(stages)+1)))
for c in range(len(fixed_x)):
    ax[0, c].imshow(fixed_x[c, 0].cpu(), cmap='gray_r', vmin=0, vmax=1)
    ax[0, c].axis('off')
for r, epoch in enumerate(stages, start=1):
    for c in range(len(fixed_x)):
        ax[r, c].imshow(snapshots[epoch][c, 0], cmap='gray_r', vmin=0, vmax=1)
        ax[r, c].axis('off')
ax[0, 0].set_ylabel('Input', fontsize=10)
for r, epoch in enumerate(stages, start=1):
    ax[r, 0].set_ylabel(f'Epoch {epoch}', fontsize=10)
fig.suptitle('Same test digits, decoded from encoder means after each training stage')
fig.tight_layout()
plt.show()

## 7. Look inside the encoder: hidden units and two output heads

The first layer still produces 128 ReLU activations. The difference is what those 128 numbers feed: **two outputs controlling position (`mu`) and two outputs controlling spread (`logvar`)**. The input-weight images below show learned coefficients, not necessarily human-interpretable features.

In [ ]:
example_i = 0  # choose 0...7
x_one = fixed_x[example_i:example_i+1]
with torch.no_grad():
    h_one = torch.relu(model.enc1(x_one.reshape(1, 784)))[0].cpu().numpy()
    mu_one, logvar_one = model.encode(x_one)
    mu_one = mu_one[0].cpu().numpy()
    std_one = torch.exp(0.5 * logvar_one[0]).cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].imshow(x_one[0, 0].cpu(), cmap='gray_r', vmin=0, vmax=1)
ax[0].set(title=f'Input (label {fixed_y[example_i].item()})')
ax[0].axis('off')
ax[1].bar(np.arange(128), h_one, width=1)
ax[1].set(title='128 hidden activations', xlabel='Unit index', ylabel='ReLU output')
ax[2].bar([0, 1], mu_one, width=.3, label='Mean mu')
ax[2].bar([.35, 1.35], std_one, width=.3, label='Std dev sigma')
ax[2].set(xticks=[.175, 1.175], xticklabels=['Dimension 1', 'Dimension 2'],
          title='Two parameters per dimension')
ax[2].legend(fontsize=9)
fig.tight_layout()
plt.show()

print('Encoder mean:', mu_one.round(3))
print('Encoder std: ', std_one.round(3))

weights = model.enc1.weight.detach().cpu().numpy()
top_units = np.argsort(h_one)[-4:][::-1]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.4))
scale = np.max(np.abs(weights[top_units]))
for j, unit in enumerate(top_units):
    axes[j].imshow(weights[unit].reshape(28, 28), cmap='coolwarm',
                   vmin=-scale, vmax=scale)
    axes[j].set_title(f'Unit {unit}')
    axes[j].axis('off')
fig.suptitle('First-layer weights of the four most active units')
fig.tight_layout()
plt.show()

## 8. Look inside one encoded distribution

For one test digit, the encoder produces a **Gaussian in the 2D plane**, not a single point. Draw several points from it and decode them all. The input remains the same: variation in output comes from random sampling of the same `q(z|x)`.

The red ellipse in the first plot marks one standard deviation along each coordinate. It is **not** a boundary beyond which samples are forbidden.

In [ ]:
from matplotlib.patches import Ellipse

model.eval()
SAMPLES_PER_DIGIT = 20
with torch.no_grad():
    mu_t, logvar_t = model.encode(x_one)
    std_t = torch.exp(0.5 * logvar_t)
    eps_t = torch.randn(SAMPLES_PER_DIGIT, 2, device=DEVICE)
    z_many = mu_t + std_t * eps_t
    imgs_many = model.decode(z_many).cpu().numpy()[:, 0]
    mean_image = model.decode(mu_t)[0, 0].cpu().numpy()

center = mu_t[0].cpu().numpy()
spread = std_t[0].cpu().numpy()
points = z_many.cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(9.5, 4))
ax[0].scatter(points[:, 0], points[:, 1], s=24, alpha=.7, label='Samples z')
ax[0].scatter(*center, marker='x', s=90, c='black', label='Mean mu')
ax[0].add_patch(Ellipse(center, width=2*spread[0], height=2*spread[1],
                        fill=False, edgecolor='tab:red', linewidth=2))
ax[0].set(xlabel='z1', ylabel='z2', title='One q(z|x) and samples')
ax[0].set_aspect('equal', adjustable='datalim')
ax[0].grid(alpha=.2)
ax[0].legend(frameon=False)
ax[1].imshow(mean_image, cmap='gray_r', vmin=0, vmax=1)
ax[1].set(title='Decoder at the mean (no sampling)')
ax[1].axis('off')
fig.tight_layout()
plt.show()

fig, axs = plt.subplots(4, 5, figsize=(7.5, 6))
for i, a in enumerate(axs.flat):
    a.imshow(imgs_many[i], cmap='gray_r', vmin=0, vmax=1)
    a.axis('off')
fig.suptitle('20 decoded samples from the distribution of ONE input digit')
fig.tight_layout()
plt.show()

**Where does the KL value come from for this digit?** Each dimension contributes a penalty for the mean's distance from zero and a penalty for variance differing from one:

\[
D_{\mathrm{KL}}=\frac12\sum_{j=1}^{2}
\left[\underbrace{\mu_j^2}_{\text{mean part}}+
\underbrace{(\sigma_j^2-1-\log\sigma_j^2)}_{\text{variance part}}\right].
\]

Both parts are nonnegative. This decomposition lets us see **what** the KL term discourages, rather than treating it as a black box.

In [ ]:
with torch.no_grad():
    mu8, logvar8 = model.encode(fixed_x)
    mean_part = 0.5 * mu8.square().sum(dim=1)
    variance_part = 0.5 * (logvar8.exp() - 1 - logvar8).sum(dim=1)

positions = np.arange(len(fixed_x))
plt.figure(figsize=(9, 3.5))
plt.bar(positions, mean_part.cpu(), label='KL: mean penalty')
plt.bar(positions, variance_part.cpu(), bottom=mean_part.cpu(),
        label='KL: variance penalty')
plt.xticks(positions, [str(int(y)) for y in fixed_y])
plt.xlabel('True digit label (not used in training)')
plt.ylabel('KL per image')
plt.title('Where the KL penalty comes from (eight test images)')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## 9. Look inside the whole latent space

We will show **three different collections of points**:

1. Encoder **means** \(\mu(x)\) for 3,000 test digits (coloured by their labels).
2. One **sample** \(z\sim q(z\mid x)\) for each of the same digits.
3. Points sampled from the **prior** \(z\sim\mathcal N(0,I)\), without any input images.

The second collection approximates the **aggregated posterior** (the distribution of encoded samples across the dataset). The standard KL objective encourages compatibility with the prior but does not force these distributions to match perfectly.

In [ ]:
@torch.no_grad()
def encode_dataset(loader):
    model.eval()
    means, stds, samples, labels = [], [], [], []
    for x, y in loader:
        mu, logvar = model.encode(x.to(DEVICE))
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        means.append(mu.cpu().numpy())
        stds.append(std.cpu().numpy())
        samples.append(z.cpu().numpy())
        labels.append(y.numpy())
    return (np.concatenate(means), np.concatenate(stds),
            np.concatenate(samples), np.concatenate(labels))

mu_test, std_test, z_post, y_test = encode_dataset(test_loader)
z_prior = np.random.default_rng(SEED).normal(size=(len(mu_test), 2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
for digit in range(10):
    idx = y_test == digit
    axes[0].scatter(mu_test[idx, 0], mu_test[idx, 1], s=7, alpha=.5, label=str(digit))
axes[1].scatter(z_post[:, 0], z_post[:, 1], s=7, alpha=.25)
axes[2].scatter(z_prior[:, 0], z_prior[:, 1], s=7, alpha=.25)
for ax, title in zip(axes, ['Encoder means (labels added)',
                            'Samples from q(z|x)', 'Samples from N(0,I)']):
    ax.set(xlabel='z1', ylabel='z2', title=title, xlim=(-4, 4), ylim=(-4, 4))
    ax.grid(alpha=.15)
    ax.set_aspect('equal', adjustable='box')
axes[0].legend(title='Digit', ncol=5, frameon=False, markerscale=2,
               loc='upper center', bbox_to_anchor=(.5, -.15))
fig.tight_layout()
plt.show()

print('Encoder means: mean=', mu_test.mean(0).round(2),
      ' std=', mu_test.std(0).round(2))
print('Encoded samples: mean=', z_post.mean(0).round(2),
      ' std=', z_post.std(0).round(2))
print('Prior target:   mean=[0, 0], std=[1, 1]')

## 10. Look inside the decoder: generation from a known distribution

Unlike the AE, the VAE was trained with an explicit prior \(\mathcal N(0,I)\). We can choose coordinates based on that known distribution. The **decoder output is the predicted mean/probability image**, not necessarily an exact sampled binary image.

First decode an evenly spaced grid in the region \([-3,3]\times[-3,3]\). The grid includes points the encoder may never have produced. Then compare prior samples with points sampled from the encoder distributions.

In [ ]:
@torch.no_grad()
def decode_points(points):
    model.eval()
    tensor = torch.as_tensor(np.asarray(points), dtype=torch.float32, device=DEVICE)
    return model.decode(tensor).cpu().numpy()[:, 0]

N_GRID = 13
xs = np.linspace(-3., 3., N_GRID)
ys = np.linspace(3., -3., N_GRID)  # high z2 at the top
points_grid = np.array([(x, y) for y in ys for x in xs], dtype=np.float32)
images_grid = decode_points(points_grid).reshape(N_GRID, N_GRID, 28, 28)
mosaic = np.concatenate([np.concatenate(row, axis=1) for row in images_grid], axis=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(mu_test[:, 0], mu_test[:, 1], s=5, alpha=.25, label='Encoder means')
ax[0].set(xlim=(-3, 3), ylim=(-3, 3), xlabel='z1', ylabel='z2',
          title='Where input means are located')
ax[0].grid(alpha=.15)
ax[0].set_aspect('equal', adjustable='box')
ax[1].imshow(mosaic, cmap='gray_r', vmin=0, vmax=1,
             extent=(-3, 3, -3, 3), origin='upper')
ax[1].set(xlabel='z1', ylabel='z2', title='Decoder evaluated on a regular grid')
fig.tight_layout()
plt.show()

In [ ]:
rng_np = np.random.default_rng(SEED)
N_SHOW = 10
prior_points = rng_np.normal(size=(N_SHOW, 2)).astype('float32')
posterior_points = z_post[rng_np.choice(len(z_post), N_SHOW, replace=False)]

fig, ax = plt.subplots(2, N_SHOW, figsize=(13, 3.4))
for row, point_set in enumerate([posterior_points, prior_points]):
    decoded = decode_points(point_set)
    for col in range(N_SHOW):
        ax[row, col].imshow(decoded[col], cmap='gray_r', vmin=0, vmax=1)
        ax[row, col].axis('off')
ax[0, 0].set_ylabel('From q(z|x)', fontsize=10)
ax[1, 0].set_ylabel('From prior', fontsize=10)
fig.suptitle('Two ways to sample points; the same trained decoder')
fig.tight_layout()
plt.show()

### Optional: compare with the exact AE from Module 2

If you saved `ae_mnist_2d.pt` in the previous notebook, place that file alongside this notebook. We load those weights and give **both decoders exactly the same prior-sampled coordinates**. If the file is absent, this cell simply skips the comparison.

This is a comparison of sampling behaviour, **not a claim that the VAE must look better on every individual sample**. Also, AE and VAE were optimized with different loss functions.

In [ ]:
class AE(nn.Module):  # unchanged AE definition from Module 2
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Linear(784, 128)
        self.enc2 = nn.Linear(128, 2)
        self.dec1 = nn.Linear(2, 128)
        self.dec2 = nn.Linear(128, 784)

    def encode(self, x):
        h = torch.relu(self.enc1(x.reshape(x.size(0), 784)))
        return self.enc2(h)

    def decode(self, z):
        h = torch.relu(self.dec1(z))
        return torch.sigmoid(self.dec2(h)).reshape(-1, 1, 28, 28)

    def forward(self, x):
        return self.decode(self.encode(x))

AE_CHECKPOINT = Path('ae_mnist_2d.pt')
if not AE_CHECKPOINT.exists():
    print('No AE checkpoint found. To compare, put ae_mnist_2d.pt next to this notebook.')
else:
    ae = AE().to(DEVICE)
    ae.load_state_dict(torch.load(AE_CHECKPOINT, map_location=DEVICE, weights_only=True))
    ae.eval()
    comparison_points = np.random.default_rng(17).normal(size=(10, 2)).astype('float32')
    with torch.no_grad():
        pts = torch.as_tensor(comparison_points, device=DEVICE)
        ae_images = ae.decode(pts).cpu().numpy()[:, 0]
    vae_images = decode_points(comparison_points)
    fig, axes = plt.subplots(2, 10, figsize=(13, 3.4))
    for col in range(10):
        axes[0, col].imshow(ae_images[col], cmap='gray_r', vmin=0, vmax=1)
        axes[1, col].imshow(vae_images[col], cmap='gray_r', vmin=0, vmax=1)
        axes[0, col].axis('off')
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('AE', fontsize=11)
    axes[1, 0].set_ylabel('VAE', fontsize=11)
    fig.suptitle('Same z ~ N(0,I), two trained decoders')
    fig.tight_layout()
    plt.show()

## 11. Interpolation: what does a straight line between two means produce?

Find two test digits. For each, take the **encoder mean**, not a random sample, to make endpoints stable. Then decode the line

\[
z(t)=(1-t)\mu_A+t\mu_B,\qquad t\in[0,1].
\]

The second row blends the **pixels** directly. These procedures answer different questions. A smooth change in pixel values does not guarantee a valid or recognizable intermediate digit.

In [ ]:
def first_test_digit(digit):
    for image, label in test_ds:
        if int(label) == digit:
            return image.unsqueeze(0).to(DEVICE)
    raise ValueError(f'No digit {digit} in the test subset')

DIGIT_A, DIGIT_B = 3, 8  # try other pairs; also try two instances of one label
x_a = first_test_digit(DIGIT_A)
x_b = first_test_digit(DIGIT_B)
model.eval()
with torch.no_grad():
    mu_a = model.encode(x_a)[0][0]
    mu_b = model.encode(x_b)[0][0]

t_values = torch.linspace(0, 1, 11, device=DEVICE)
z_path = (1 - t_values[:, None]) * mu_a + t_values[:, None] * mu_b
latent_images = decode_points(z_path.cpu().numpy())
pixel_images = np.stack([((1-t)*x_a + t*x_b)[0, 0].cpu().numpy()
                         for t in t_values])

fig, ax = plt.subplots(2, len(t_values), figsize=(13, 3.6))
for i, t in enumerate(t_values):
    ax[0, i].imshow(latent_images[i], cmap='gray_r', vmin=0, vmax=1)
    ax[1, i].imshow(pixel_images[i], cmap='gray_r', vmin=0, vmax=1)
    ax[0, i].set_title(f'{t.item():.1f}', fontsize=9)
    ax[0, i].axis('off')
    ax[1, i].axis('off')
ax[0, 0].set_ylabel('Latent', fontsize=10)
ax[1, 0].set_ylabel('Pixels', fontsize=10)
fig.suptitle(f'Interpolation between encoded means: digit {DIGIT_A} to {DIGIT_B}')
fig.tight_layout()
plt.show()

path = z_path.cpu().numpy()
fig, ax = plt.subplots(figsize=(6.3, 4.5))
ax.scatter(mu_test[:, 0], mu_test[:, 1], s=5, alpha=.15, label='Test encoder means')
ax.plot(path[:, 0], path[:, 1], 'o-', c='black', lw=1.5, ms=3,
        label='Interpolation')
ax.annotate(str(DIGIT_A), path[0], xytext=(7, 7), textcoords='offset points')
ax.annotate(str(DIGIT_B), path[-1], xytext=(7, 7), textcoords='offset points')
ax.set(xlabel='z1', ylabel='z2', title='A straight line in latent coordinates')
ax.grid(alpha=.15)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 12. Investigate the VAE yourself

**A. One image, many possible codes.** Change `example_i` in sections 7–8. Which image produces a wide distribution? Does decoding 20 samples from one image preserve the digit's identity?

**B. Reconstruction versus KL.** Change `BETA` to `0.1`, `1`, or `4`, **reinitialize the model**, and rerun training. Record test MSE, test KL, and the prior-sample grid. What improves and what deteriorates? *(Changing only `BETA` after training does not retrain anything.)*

**C. AE versus VAE.** If your AE checkpoint is available, compare the two models at the same ten points sampled from `N(0,I)`. Explain why the VAE has a reason to perform better at this task. Is the result guaranteed?

**D. Geometry.** Find a pair of digits whose interpolation appears meaningful and another pair whose interpolation is questionable. Where do the paths travel relative to the cloud of encoder means? Does passing through a sparse region always imply failure?

**E. Connect to the paper.** In Kingma & Welling (2013/2014), identify the equations corresponding to the two encoder heads, the ELBO, and reparameterization. Which code lines implement each mathematical step?

Select a point below and examine its neighbourhood. This is intentionally similar to the final exercise in the AE notebook.

In [ ]:
MY_Z = (0.0, 0.0)  # change to coordinates of interest
STEP = 0.5
neighbourhood = np.array([(MY_Z[0] + i*STEP, MY_Z[1] + j*STEP)
                          for j in [1, 0, -1] for i in [-1, 0, 1]],
                         dtype=np.float32)
imgs = decode_points(neighbourhood)
fig, ax = plt.subplots(3, 3, figsize=(5.8, 5.8))
for a, p, im in zip(ax.flat, neighbourhood, imgs):
    a.imshow(im, cmap='gray_r', vmin=0, vmax=1)
    a.set_title(f'({p[0]:.1f}, {p[1]:.1f})', fontsize=9)
    a.axis('off')
fig.suptitle('Move around a chosen point in latent space')
fig.tight_layout()
plt.show()

## 13. What changed from AE to VAE?

| Component | AE notebook | VAE notebook |
|:--|:--|:--|
| Encoder output | One point `z` | Two means `mu` and two log-variances `logvar` |
| Bottleneck | Deterministic point | Random sample `z = mu + std * eps` |
| Reconstruction objective | Mean squared pixel error | Cross-entropy pixel reconstruction score |
| Additional objective | None | KL divergence toward `N(0,I)` |
| Decoder architecture | `2 → 128 → 784` | **Unchanged** |
| How to sample | No specified prior | Sample `z ~ N(0,I)` |

**Conclusion to test, not assume:** the VAE deliberately links encoder distributions to a known prior, making *sampling* a trained use case. It can trade some reconstruction accuracy for that property; smooth or meaningful interpolation is still not guaranteed.

Save the parameters below. Rerunning training produces a new model; saving allows you to revisit exactly the model you inspected.

In [ ]:
torch.save(model.state_dict(), 'vae_mnist_2d.pt')
print('Saved model weights to vae_mnist_2d.pt')